# [LAB] A2A로 Researcher에게 일 맡기기

검색 도구를 쓰는 Researcher Agent를 A2A 서버로 띄우고, 이 노트북을 클라이언트로 삼아 조사를 맡깁니다. 받은 조사 결과로는 이 노트북의 모델이 블로그 글을 씁니다. 보기 중 하나를 골라 주석을 풀어 코드를 완성하고, 완성된 코드는 실행하며 값을 바꿔 봅니다.

- **2장 Researcher Agent를 A2A 서버로 띄우기**: 셀들이 서버 파일을 씁니다. 다섯 곳에서 보기를 고르고, 터미널에서 서버를 띄웁니다.
- **3장 Researcher에게 일을 맡기는 클라이언트**: 요청 메시지를 만드는 곳 한 군데에서 보기를 고르고, Agent Card 확인 → 위임 → 글쓰기를 차례로 실행합니다.

셀은 위에서부터 차례로 실행합니다. 서버는 터미널에서 켜 둔 채로 3장을 진행합니다. 요청 한 번에 모델을 두세 번 호출하므로 결과가 나오기까지 수십 초가 걸립니다.

**목차**

- 1. 실습 방법
- 2. Researcher Agent를 A2A 서버로 띄우기
  - 2.1 불러오기
  - 2.2 검색 도구와 LLM 준비
  - 2.3 agent 노드
  - 2.4 그래프 연결과 컴파일
  - 2.5 A2A AgentExecutor
  - 2.6 Agent Card
  - 2.7 Starlette 앱 조립
  - 2.8 AI에게 물어보기
- 3. Researcher에게 일을 맡기는 클라이언트
  - 3.1 call_researcher 함수
  - 3.2 Agent Card 확인
  - 3.3 Researcher에게 위임
  - 3.4 Writer가 글 작성
  - 3.5 AI에게 물어보기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Researcher Agent를 A2A 서버로 띄우기

검색 도구를 쓰는 작은 LangGraph Agent(Researcher)를 만들고, 다른 Agent가 요청할 수 있도록 A2A 서버로 감쌉니다. 서버는 노트북 셀에서 돌리지 않고 파일로 쓴 뒤 터미널에서 띄웁니다.

아래 셀은 모두 서버 파일 `a2a_server.py`의 한 부분입니다. 첫 셀이 파일을 새로 쓰고(`%%writefile`), 다음 셀부터는 뒤에 이어 씁니다(`%%writefile -a`). 그래서 **위에서부터 차례로** 실행해야 하고, 고른 보기를 바꿨다면 **첫 셀(2.1)부터 다시** 실행해 파일을 새로 씁니다. 서버가 켜져 있으면 Ctrl+C로 끄고 다시 띄워야 바뀐 내용이 반영됩니다.

셀을 실행하면 파일만 써지고 코드는 아직 실행되지 않습니다. 맞는 보기를 골랐는지는 서버를 띄우고 3장의 클라이언트로 요청을 보내야 보입니다. 그래서 2장에서는 다섯 곳을 모두 고른 뒤 서버를 띄우고, 3장에서 결과를 보고 돌아와 고칩니다.

고를 곳은 다섯 군데입니다. Agent Card 셀은 완성된 코드라 실행만 합니다.

**왜 배우나:** 안쪽 Agent는 지금까지 만든 LangGraph Agent와 같습니다. 여기에 AgentExecutor, Agent Card, RequestHandler 세 가지를 더하면 다른 Agent가 요청할 수 있는 서버가 됩니다.

### 2.1 불러오기

In [ ]:
%%writefile a2a_server.py
"""
A2A 서버: LangGraph Agent를 A2A 프로토콜로 노출하기

A2A(Agent-to-Agent)는 Google이 제안한 에이전트 간 통신 표준이다.
에이전트를 HTTP 서비스로 노출하면 프레임워크에 관계없이 (LangGraph, CrewAI, AutoGen 등)
서로 통신할 수 있다.
"""

import asyncio
import uuid
from typing import Annotated

import a2a.types as t
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers.default_request_handler_v2 import (
    DefaultRequestHandlerV2,
)
from a2a.server.routes.agent_card_routes import create_agent_card_routes
from a2a.server.routes.jsonrpc_routes import create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.events.in_memory_queue_manager import InMemoryQueueManager
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from starlette.applications import Starlette
from starlette.routing import Route
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

**성공 기준**

- 🟢 셀 아래에 `Writing a2a_server.py` 또는 `Overwriting a2a_server.py` 한 줄이 출력되었다

### 2.2 검색 도구와 LLM 준비

**✅ CHOOSE_HERE: search_web 도구를 LLM과 ToolNode에 연결해 주세요.**

Researcher가 쓸 모델과 도구 실행 노드를 준비합니다. 모델에는 `search_web` 도구를 연결해 도구 호출을 요청할 수 있게 하고, 그 요청을 실제로 실행할 `ToolNode`도 같은 도구로 만듭니다. 아래 셀에서 쓸 이름은 `llm`과 `tool_node`입니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되고, 서버를 띄울 때 오류 없이 켜진다
- 🟢 3장에서 위임했을 때 서버 터미널에 `[search_web] 검색어: …` 줄이 나온다(모델이 검색 없이 답하면 안 나올 수 있으니 한두 번 더 위임해 봅니다)
- ⛔ 서버를 띄우자마자 터미널에 `ValueError`나 `TypeError`가 나오고 끝난다 → 고른 보기를 다시 주석 처리하고 다른 보기를 고릅니다. 2.1 셀부터 차례로 다시 실행하고 서버를 다시 띄웁니다
- ⛔ 세 번 위임해도 서버 터미널에 `[search_web]` 줄이 한 번도 나오지 않는다 → 모델이 도구를 알고 있는지 봅니다. 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고 서버를 다시 띄웁니다

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── 내부 LangGraph Agent ──────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색한다."""
    print(f"[search_web] 검색어: {query}")
    return f'"{query}" 검색 결과: 해당 주제에 대한 최신 정보 (데모 데이터)'


class State(TypedDict):
    messages: Annotated[list, add_messages]


# ✅ CHOOSE_HERE: search_web 도구를 LLM과 ToolNode에 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# llm = get_llm().bind_tools([search_web])    # 1️⃣ 도구 연결 + 도구 하나로 ToolNode
# tool_node = ToolNode(search_web)            # 1️⃣
#
# llm = get_llm()                             # 2️⃣ 모델만 준비 + ToolNode
# tool_node = ToolNode([search_web])          # 2️⃣
#
# llm = get_llm().bind_tools([search_web])    # 3️⃣ 도구 연결 + 같은 도구 목록으로 ToolNode
# tool_node = ToolNode([search_web])          # 3️⃣
#
# llm = get_llm().bind_tools([search_web()])  # 4️⃣ 도구를 불러 결과를 연결
# tool_node = ToolNode([search_web])          # 4️⃣

<details><summary>정답 보기</summary>

**정답: 3️⃣**

- 1️⃣ `ToolNode`에는 도구 목록을 넘겨야 합니다. 목록 없이 도구 하나를 넘기면 서버를 띄우자마자 `ValueError: The first argument must be a string or a callable …`가 나고 끝납니다.
- 2️⃣ 오류 없이 서버가 켜지고 리서치 결과 글도 나옵니다. 하지만 모델이 도구를 몰라 검색하지 않습니다. 서버 터미널에 `[search_web]` 줄이 한 번도 나오지 않습니다.
- 3️⃣ `bind_tools`로 모델에 도구를 알려 주고, 같은 도구 목록으로 도구를 실제로 실행할 `ToolNode`를 만듭니다. 위임하면 서버 터미널에 `[search_web]` 줄이 나옵니다.
- 4️⃣ 도구 뒤에 괄호를 붙이면 그 자리에서 부릅니다. 서버를 띄우자마자 `TypeError: 'StructuredTool' object is not callable`이 나고 끝납니다.

</details>

### 2.3 agent 노드

**✅ CHOOSE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

`agent_node`는 Researcher가 한 번 생각하는 단계입니다. 지금까지 쌓인 메시지를 도구가 연결된 `llm`에 보내고, 받은 응답을 상태의 `messages`에 더하도록 돌려줍니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되고, 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ 3.3 위임 셀에서 `InternalError`가 난다 → 서버 터미널의 Traceback을 봅니다. 고른 보기를 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고, 서버를 다시 띄웁니다
- ⛔ `리서치 결과:` 아래에 보낸 주제만 그대로 나온다 → 모델의 응답을 `messages`에 담아 돌려주는지 봅니다
- ⛔ 세 번 위임해도 서버 터미널에 `[search_web]` 줄이 한 번도 나오지 않는다 → 도구가 연결된 `llm`을 호출하는지 봅니다

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
def agent_node(state: State) -> dict:
    # ✅ CHOOSE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
    # response = llm.invoke(state["messages"])        # 1️⃣ 도구가 연결된 llm에 메시지 목록을 넘김
    # return {"messages": [response]}                 # 1️⃣
    #
    # response = llm.invoke(state)                    # 2️⃣ State 전체를 넘김
    # return {"messages": [response]}                 # 2️⃣
    #
    # response = get_llm().invoke(state["messages"])  # 3️⃣ 새로 만든 모델에 메시지를 넘김
    # return {"messages": [response]}                 # 3️⃣
    #
    # response = llm.invoke(state["messages"])        # 4️⃣ 답 메시지를 그대로 돌려줌
    # return response                                 # 4️⃣
    pass  # 그대로 둡니다

<details><summary>정답 보기</summary>

**정답: 1️⃣**

- 1️⃣ 2.2에서 도구를 연결한 `llm`에 메시지 목록을 넘기고, 답을 대괄호로 감싸 `messages`에 담습니다. `add_messages`가 기존 목록 뒤에 쌓습니다.
- 2️⃣ State 딕셔너리를 통째로 넘깁니다. 서버는 켜지지만 위임하면 서버 터미널에 `ValueError: Invalid input type <class 'dict'>`가 나오고, 3.3에서 `InternalError`가 납니다.
- 3️⃣ 도구가 연결되지 않은 새 모델이라 검색하지 않습니다. 오류 없이 리서치 결과 글이 나오지만 서버 터미널에 `[search_web]` 줄이 한 번도 나오지 않습니다.
- 4️⃣ 딕셔너리가 아니라서 State가 바뀌지 않습니다. 오류 없이 3.3의 `리서치 결과:` 아래에 보낸 주제만 그대로 나옵니다.

</details>

### 2.4 그래프 연결과 컴파일

**✅ CHOOSE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

노드 두 개(`agent`, `tools`)와 시작 엣지는 이미 있습니다. agent 다음에는 도구 호출 요청이 있으면 tools로, 없으면 끝으로 가게 나누고, tools를 실행한 뒤에는 다시 agent로 돌아오게 잇습니다. 이 연결이 빠져도 답은 그럴듯하게 나올 수 있습니다. 그래서 서버 터미널의 `[search_web]` 줄로 도구가 실제로 호출되었는지 확인합니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되었다
- 🟢 3장에서 위임했을 때 서버 터미널에 `[search_web] 검색어: …` 줄이 나오고, 리서치 결과 글이 나온다(모델이 검색 없이 답하면 안 나올 수 있으니 한두 번 더 위임해 봅니다)
- ⛔ 서버를 띄우자마자 터미널에 `TypeError`가 나오고 끝난다 → 고른 보기를 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고 서버를 다시 띄웁니다
- ⛔ `리서치 결과:` 아래가 비어 있다 → 모델이 도구 호출을 요청했는데 tools로 가지 못하고 끝났습니다
- ⛔ `리서치 결과:` 아래에 `… 검색 결과: 해당 주제에 대한 최신 정보 (데모 데이터)` 한 줄만 나온다 → 도구를 실행한 뒤 agent로 돌아가지 못하고 끝났습니다

**✅ CHOOSE_HERE: LangGraph Agent를 컴파일해 주세요.**

연결을 마친 그래프를 실행할 수 있는 Agent로 만듭니다. 아래 AgentExecutor가 이 Agent를 `langgraph_agent`라는 이름으로 호출합니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 서버를 띄울 때 오류 없이 켜지고, 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ 서버를 띄우자마자 터미널에 `ValueError: Graph must have an entrypoint`가 나오고 끝난다 → 고른 보기를 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고 서버를 다시 띄웁니다
- ⛔ 3.3에서 위임하면 `InternalError`가 나고 `… has no attribute 'ainvoke'`가 보인다 → 컴파일한 Agent를 담았는지 봅니다. 바꾼 뒤 2.1 셀부터 다시 실행 → 서버 다시 띄우기 → 3.3 위임 셀 다시 실행 순서로 확인합니다

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
# ✅ CHOOSE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# builder.add_conditional_edges("agent", tools_condition)    # 1️⃣ 도구 호출 여부로 분기만
#
# builder.add_edge("agent", END)                             # 2️⃣ agent → END + tools → agent
# builder.add_edge("tools", "agent")                         # 2️⃣
#
# builder.add_conditional_edges("agent", tools_condition())  # 3️⃣ tools_condition을 불러 결과를 넘김
# builder.add_edge("tools", "agent")                         # 3️⃣
#
# builder.add_conditional_edges("agent", tools_condition)    # 4️⃣ 도구 호출 여부로 분기 + tools → agent
# builder.add_edge("tools", "agent")                         # 4️⃣
# ✅ CHOOSE_HERE: LangGraph Agent를 컴파일해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# langgraph_agent = builder.compile              # 1️⃣ 괄호 없이 compile을 담음
# langgraph_agent = builder.compile()            # 2️⃣ 지금까지 조립한 builder를 컴파일
# langgraph_agent = StateGraph(State).compile()  # 3️⃣ 새 그래프를 컴파일
# langgraph_agent = builder                      # 4️⃣ builder를 그대로 담음

<details><summary>정답 보기</summary>

**tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

**정답: 4️⃣**

- 1️⃣ tools를 실행한 뒤 갈 곳이 없어 거기서 끝납니다. 오류 없이 3.3의 `리서치 결과:` 아래에 `"…" 검색 결과: 해당 주제에 대한 최신 정보 (데모 데이터)` 한 줄만 나옵니다. 모델이 결과를 보고 글을 쓸 기회가 없습니다.
- 2️⃣ agent 다음은 언제나 끝입니다. 모델이 검색을 요청하면 도구를 실행하지 못하고 끝나, 오류 없이 3.3의 `리서치 결과:` 아래가 비어 있습니다.
- 3️⃣ 괄호를 붙이면 State 없이 그 자리에서 함수를 부릅니다. 서버를 띄우자마자 `TypeError: tools_condition() missing 1 required positional argument: 'state'`가 나고 끝납니다.
- 4️⃣ 도구 호출 요청이 있으면 tools로, 없으면 끝으로 갑니다. tools를 실행한 뒤에는 agent로 돌아가 검색 결과를 보고 답을 이어 씁니다. 이 되돌아가는 엣지가 순환의 고리입니다.

**LangGraph Agent를 컴파일해 주세요.**

**정답: 2️⃣**

- 1️⃣ 괄호가 없어 컴파일하지 않고 `compile` 함수 자체를 담습니다. 서버는 켜지지만 위임하면 3.3에서 `InternalError: 'function' object has no attribute 'ainvoke'`가 납니다.
- 2️⃣ 조립한 설계도(`builder`)를 실행할 수 있는 Agent로 만들어 `langgraph_agent`에 담습니다. AgentExecutor가 이 이름으로 호출합니다.
- 3️⃣ 노드가 하나도 없는 새 그래프를 컴파일합니다. 서버를 띄우자마자 `ValueError: Graph must have an entrypoint`가 나고 끝납니다.
- 4️⃣ `builder`는 설계도라서 실행할 수 없습니다. 서버는 켜지지만 위임하면 3.3에서 `InternalError: 'StateGraph' object has no attribute 'ainvoke'`가 납니다.

</details>

### 2.5 A2A AgentExecutor

**✅ CHOOSE_HERE: 내부 LangGraph Agent를 비동기로 호출하고 최종 답변을 추출해 주세요.**

AgentExecutor는 A2A 요청을 실제 Agent 실행으로 잇습니다. 요청 글(`user_text`)을 사람 메시지로 감싸 `langgraph_agent`를 **비동기로** 실행하고, 결과 메시지 목록의 마지막 내용을 `answer`에 담습니다. 바로 아래 코드가 `answer`를 A2A Message로 감싸 돌려보냅니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되고, 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ 3.3에서 위임하면 `InternalError`가 난다 → 서버 터미널의 Traceback을 봅니다. 고른 보기를 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고 서버를 다시 띄운 뒤, 3.3 위임 셀로 확인합니다
- ⛔ `리서치 결과:` 아래에 보낸 주제만 그대로 나온다 → 결과 메시지 목록의 어느 메시지를 꺼냈는지 봅니다

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── A2A AgentExecutor ─────────────────────────────────────────────────────────
# AgentExecutor.execute(): RequestContext에서 사용자 메시지를 읽어
# EventQueue에 결과(Message)를 발행한다.
#
# a2a-sdk 1.x는 a2a.types의 타입(t.Message, t.Part 등)으로 메시지를 만든다.


class ResearcherExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input(): 사용자 메시지의 텍스트 파트를 모아 문자열로 반환
        user_text = context.get_user_input() or ""

        # ✅ CHOOSE_HERE: 내부 LangGraph Agent를 비동기로 호출하고 최종 답변을 추출해 주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
        # result = langgraph_agent.ainvoke(                   # 1️⃣ await 없이 ainvoke
        #     {"messages": [HumanMessage(content=user_text)]}  # 1️⃣
        # )                                                   # 1️⃣
        # answer = result["messages"][-1].content             # 1️⃣
        #
        # result = await langgraph_agent.ainvoke(             # 2️⃣ 첫 메시지의 내용을 꺼냄
        #     {"messages": [HumanMessage(content=user_text)]}  # 2️⃣
        # )                                                   # 2️⃣
        # answer = result["messages"][0].content              # 2️⃣
        #
        # result = await langgraph_agent.ainvoke(             # 3️⃣ 마지막 메시지를 통째로 꺼냄
        #     {"messages": [HumanMessage(content=user_text)]}  # 3️⃣
        # )                                                   # 3️⃣
        # answer = result["messages"][-1]                     # 3️⃣
        #
        # result = await langgraph_agent.ainvoke(             # 4️⃣ await로 실행 + 마지막 메시지의 내용
        #     {"messages": [HumanMessage(content=user_text)]}  # 4️⃣
        # )                                                   # 4️⃣
        # answer = result["messages"][-1].content             # 4️⃣

        # t.Message: A2A 응답 메시지
        # t.Part(text=...): 텍스트 파트 (1.x에서는 TextPart 대신 Part.text 직접 사용)
        # 결과를 A2A Message로 감싸 EventQueue에 발행한다
        await event_queue.enqueue_event(
            t.Message(
                role=t.Role.ROLE_AGENT,
                parts=[t.Part(text=answer)],
                task_id=context.task_id,
                context_id=context.context_id,
                message_id=str(uuid.uuid4()),
            )
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

<details><summary>정답 보기</summary>

**정답: 4️⃣**

- 1️⃣ `await`가 없으면 실행하지 않고 실행 예약(coroutine)만 받습니다. 위임하면 서버 터미널에 `TypeError: 'coroutine' object is not subscriptable`이 나오고, 3.3에서 `InternalError`가 납니다.
- 2️⃣ 첫 메시지는 내가 보낸 질문입니다. 검색까지 다 하고도 오류 없이 3.3의 `리서치 결과:` 아래에 보낸 주제만 그대로 나옵니다.
- 3️⃣ 글이 아니라 메시지 객체라서 `t.Part(text=answer)`에 넣을 수 없습니다. 위임하면 서버 터미널에 `TypeError: bad argument type for built-in operation`이 나오고, 3.3에서 `InternalError`가 납니다.
- 4️⃣ 요청 글을 사람 메시지로 감싸 `await`로 비동기 실행하고, 마지막 메시지(모델의 최종 답)의 글을 `answer`에 담습니다.

</details>

### 2.6 Agent Card

**✍️ CHANGE_HERE: Agent Card의 설명 문구를 바꿔 봅니다**

Agent Card는 다른 Agent가 가장 먼저 읽는 자기소개입니다. 처음에는 그대로 두고 끝까지 진행합니다. 3장까지 해 본 뒤 돌아와 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 문구를 바꿉니다(예: "주제를 조사해 핵심만 세 줄로 정리합니다."). 바꾼 뒤에는 **2.1 셀부터 이 장의 셀을 차례로 다시 실행**하고, 서버 터미널에서 Ctrl+C로 끈 뒤 다시 띄웁니다.

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── Agent Card ────────────────────────────────────────────────────────────────
# Agent Card: 이 에이전트가 무엇을 할 수 있는지 설명하는 메타데이터.
# 클라이언트는 /.well-known/agent-card.json에서 이를 읽어 에이전트를 발견(discover)한다.

# 이름, 설명, 입출력 모드, capability, skill 정보를 적는다
agent_card = t.AgentCard(
    name="Researcher Agent",
    # ✍️ CHANGE_HERE: Agent Card의 설명 문구를 바꿔 봅니다
    description="주어진 주제를 웹에서 조사하고 요약 정보를 제공합니다.",
    version="1.0.0",
    capabilities=t.AgentCapabilities(streaming=False),
    default_input_modes=["text"],
    default_output_modes=["text"],
    supported_interfaces=[t.AgentInterface(url="http://localhost:8001")],
    skills=[
        t.AgentSkill(
            id="web_research",
            name="웹 리서치",
            description="주제를 검색해 핵심 정보를 수집하고 정리합니다.",
            tags=["research", "web"],
            examples=["AI Agent 기술 트렌드 조사", "LangGraph 최신 기능 알려줘"],
        )
    ],
)

### 2.7 Starlette 앱 조립

In [ ]:
%%writefile -a a2a_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── Starlette 앱 조립 ─────────────────────────────────────────────────────────

queue_manager = InMemoryQueueManager()
task_store = InMemoryTaskStore()

request_handler = DefaultRequestHandlerV2(
    # AgentExecutor, TaskStore, AgentCard, QueueManager를 연결한다
    agent_executor=ResearcherExecutor(),
    task_store=task_store,
    queue_manager=queue_manager,
    agent_card=agent_card,
)

# Agent Card 조회 라우트와 JSON-RPC 라우트를 만든다
routes: list[Route] = [
    *create_agent_card_routes(agent_card),
    *create_jsonrpc_routes(request_handler, rpc_url="/"),
]

app = Starlette(routes=routes)

if __name__ == "__main__":
    print("Researcher A2A 서버 시작: http://localhost:8001")
    print("Agent Card: http://localhost:8001/.well-known/agent-card.json")
    # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
    uvicorn.run(app, host="127.0.0.1", port=8001)

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되었다

2.1 셀부터 이 안내 바로 위 셀까지 차례로 실행하면 이 폴더에 서버 파일이 만들어집니다(보기를 먼저 고릅니다). 서버는 터미널에서 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/a2a_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다.
4. 서버 파일을 고쳤다면 터미널에서 **Ctrl+C**로 끄고, 2.1 셀부터 이 안내 바로 위 셀까지 차례로 모두 다시 실행한 뒤 같은 명령으로 다시 띄웁니다. 고친 셀 하나만 다시 실행하면 파일 끝에 덧붙기만 해서 반영되지 않습니다.

**볼 지점**
- 서버를 띄운 터미널에 먼저 아래 두 줄이 나옵니다. 서버 파일 맨 끝의 `print`가 출력한 것입니다. 그 아래에 서버 프로그램(uvicorn)이 `INFO:`로 시작하는 줄을 이어서 출력하고, 마지막 줄에 `Uvicorn running on`과 `127.0.0.1:8001`이 보입니다(`127.0.0.1`은 이 PC에서 오는 요청만 기다린다는 뜻입니다).

```
Researcher A2A 서버 시작: http://localhost:8001
Agent Card: http://localhost:8001/.well-known/agent-card.json
```

- 시작 문구 위에 `DeprecationWarning`이 나옵니다. 슬라이드대로 `queue_manager`를 넘기지만, 지금 설치된 버전은 이벤트를 안에서 따로 관리해 이 값을 쓰지 않는다는 안내입니다. 서버 동작에는 영향이 없으니 넘어갑니다.
- 서버는 요청이 올 때까지 기다리기만 합니다. 모델은 요청이 들어올 때 호출됩니다.
- 3장에서 요청을 보내면 이 터미널에 `"POST / HTTP/1.1" 200 OK`로 끝나는 접속 기록이 나옵니다. 모델이 검색하기로 하면 그 앞에 `[search_web] 검색어: …` 줄도 나옵니다. 검색 여부는 모델이 정하므로 없는 때도 있습니다.
- 클라이언트에서 오류가 나면 **이 터미널부터** 봅니다. 서버 코드에서 난 오류(Traceback)가 여기에 나옵니다.

**성공 기준**

- 🟢 터미널 마지막 줄에 `Uvicorn running on`과 `127.0.0.1:8001`이 보이고, 명령이 끝나지 않고 기다린다
- ⛔ `NameError`·`SyntaxError`·`TypeError`·`ValueError`가 나오고 바로 끝난다 → 오류가 가리키는 줄이 있는 셀에서 고른 보기를 다시 봅니다. 다른 보기로 바꾼 뒤 2.1 셀부터 차례로 다시 실행하고 다시 띄웁니다
- ⛔ `address already in use` 또는 `10048`(Windows)이 나온다 → 8001 포트를 쓰는 서버가 이미 켜져 있습니다. 앞서 띄운 서버 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

### 2.8 AI에게 물어보기

**프롬프트 힌트** `ResearcherExecutor`, `request_handler`, `routes` 셀을 붙여 넣고, 요청 한 건이 들어와 응답이 나갈 때까지 어떤 객체를 어떤 순서로 거치는지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 라우트(JSON-RPC) → RequestHandler → AgentExecutor → 내부 LangGraph Agent → EventQueue 순서를 짚었다
- 🟢 Agent Card 라우트는 카드 조회용이고 작업 요청은 JSON-RPC 라우트로 들어온다는 점을 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다
- ⛔ `A2AStarletteApplication` 같은 이 코드에 없는 이름으로 설명한다 → 붙여 넣은 코드에 있는 이름으로만 설명해 달라고 합니다

## 3. Researcher에게 일을 맡기는 클라이언트

이제 노트북이 클라이언트가 되어, 터미널에 띄운 Researcher 서버에 A2A로 조사를 맡깁니다. 순서는 슬라이드와 같습니다. Agent Card를 읽고, 그 정보로 Client를 만들고, 요청을 `t.Message`에 담아 보냅니다. 받은 조사 결과로는 이 노트북의 모델(Writer)이 블로그 글을 씁니다.

**이 장을 시작하기 전에 2장의 서버가 터미널에서 켜져 있어야 합니다.**

**왜 배우나:** 조사는 원격 Agent에게 맡기고 글쓰기는 여기서 직접 하는 분업입니다. 맡기는 쪽은 Researcher가 안에서 어떻게 만들어졌는지 몰라도 됩니다.

준비 코드입니다. A2A 클라이언트 도구를 불러오고, Researcher 서버 주소(`RESEARCHER_URL`)를 정합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
import asyncio
import uuid

import a2a.types as t
import httpx
from a2a.client.base_client import BaseClient
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.client.transports.jsonrpc import JsonRpcTransport
from a2a.utils.constants import PROTOCOL_VERSION_1_0, VERSION_HEADER
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()

RESEARCHER_URL = "http://127.0.0.1:8001"

### 3.1 call_researcher 함수

**✅ CHOOSE_HERE: t.SendMessageRequest를 생성해주세요.**

`call_researcher`는 Agent Card 조회 → Client 생성 → 메시지 전송 → 응답에서 텍스트 꺼내기를 한 함수에 담았습니다. 고를 곳은 세 번째 단계입니다. 받은 `query`를 사용자 역할의 `t.Message`에 텍스트 파트로 담고, 메시지마다 고유한 id를 붙여 `SendMessageRequest`로 감싼 `req`를 만듭니다. 2장의 "A2A AgentExecutor" 셀에서 서버가 답을 보낼 때 쓴 `t.Message(...)`와 비교해 봅니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다. 실제 호출은 아래 "Researcher에게 위임" 셀에서 합니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 이 셀이 오류 없이 끝나고 아무것도 출력하지 않는다
- 🟢 아래 "Researcher에게 위임" 셀에서 리서치 결과 글이 나온다
- ⛔ "Researcher에게 위임" 셀에서 `TypeError`나 `InvalidParamsError`가 나고 서버 터미널에는 새 Traceback이 없다 → 고른 보기를 다른 보기로 바꾼 뒤 이 셀을 다시 실행하고, 3.3 위임 셀을 다시 실행합니다(서버는 다시 띄우지 않아도 됩니다)
- ⛔ 리서치 결과가 보낸 주제와 관계없는 글이다(예: 무엇을 검색할지 되묻는다) → 서버 터미널의 `[search_web] 검색어:` 줄을 보고, 무엇을 보냈는지 확인합니다

In [ ]:
async def call_researcher(query: str) -> str:
    """A2A 프로토콜로 Researcher Agent에게 조사를 요청한다.

    A2A-Version 헤더를 명시해야 서버가 1.x 형식 요청임을 인식한다.
    헤더가 없으면 서버는 레거시(0.3) 버전으로 가정하여 거부한다.
    """
    headers = {VERSION_HEADER: PROTOCOL_VERSION_1_0}
    async with httpx.AsyncClient(headers=headers, timeout=60.0) as http:
        # 1. Agent Card 조회: 에이전트가 어떤 기능을 제공하는지 확인
        resolver = A2ACardResolver(http, RESEARCHER_URL)
        card = await resolver.get_agent_card()

        # 2. JSON-RPC 트랜스포트로 클라이언트 생성
        transport = JsonRpcTransport(http, card, f"{RESEARCHER_URL}/")
        client = BaseClient(card, ClientConfig(), transport, [])

        # 3. 메시지 전송
        # ✅ CHOOSE_HERE: t.SendMessageRequest를 생성해주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
        # req = t.SendMessageRequest(                # 1️⃣ 사용자 메시지 + 텍스트 파트 + 고유 id
        #     message=t.Message(                     # 1️⃣
        #         role=t.Role.ROLE_USER,             # 1️⃣
        #         parts=[t.Part(text=query)],        # 1️⃣
        #         message_id=str(uuid.uuid4()),      # 1️⃣
        #     )                                      # 1️⃣
        # )                                          # 1️⃣
        #
        # req = t.SendMessageRequest(                # 2️⃣ 사용자 메시지 + 텍스트 파트
        #     message=t.Message(                     # 2️⃣
        #         role=t.Role.ROLE_USER,             # 2️⃣
        #         parts=[t.Part(text=query)],        # 2️⃣
        #     )                                      # 2️⃣
        # )                                          # 2️⃣
        #
        # req = t.SendMessageRequest(                # 3️⃣ 텍스트 파트에 "query" 글자
        #     message=t.Message(                     # 3️⃣
        #         role=t.Role.ROLE_USER,             # 3️⃣
        #         parts=[t.Part(text="query")],      # 3️⃣
        #         message_id=str(uuid.uuid4()),      # 3️⃣
        #     )                                      # 3️⃣
        # )                                          # 3️⃣
        #
        # req = t.SendMessageRequest(message=query)  # 4️⃣ query 글을 바로 message에 넣음

        async for response in client.send_message(req):
            # StreamResponse.message: 에이전트가 반환한 Message 객체
            if response.HasField("message"):
                return "".join(
                    part.text for part in response.message.parts if part.text
                )

    return "응답을 받지 못했습니다."

<details><summary>정답 보기</summary>

**정답: 1️⃣**

- 1️⃣ 받은 `query`를 사용자 역할의 메시지에 텍스트 파트로 담고, 고유한 `message_id`를 붙여 요청으로 감쌉니다.
- 2️⃣ `message_id`가 없어 서버가 요청을 받지 않습니다. 3.3에서 `InvalidParamsError: Validation failed`가 나고, 서버 터미널에는 Traceback이 없습니다.
- 3️⃣ 따옴표 때문에 변수 값이 아니라 `query`라는 글자를 보냅니다. 오류 없이 서버 터미널에 `[search_web] 검색어: query`가 나오고, 리서치 결과는 주제와 관계없는 글(무엇을 검색할지 되묻는 글 등)이 나옵니다.
- 4️⃣ `message`에는 `t.Message`를 넣어야 합니다. 3.3에서 `TypeError: … expected <class 'a2a_pb2.Message'> got <class 'str'>`이 납니다. 서버까지 가지 않아 서버 터미널에는 아무것도 나오지 않습니다.

</details>

### 3.2 Agent Card 확인

In [ ]:
# ── Agent Card 확인 ────────────────────────────────────────────────────────
async with httpx.AsyncClient() as http:
    resp = await http.get(f"{RESEARCHER_URL}/.well-known/agent-card.json")
    card = resp.json()

print("=== Researcher Agent Card ===")
print(f"이름   : {card['name']}")
print(f"설명   : {card['description']}")
print(f"스킬   : {[s['name'] for s in card.get('skills', [])]}")

**성공 기준**

- 🟢 `이름   : Researcher Agent`, `설명   : …`, `스킬   : ['웹 리서치']` 세 줄이 출력되었다
- 🟢 (2장에서 설명 문구를 바꿨다면) 설명 줄이 바꾼 문구로 나온다
- ⛔ `ConnectError`(연결 거부)가 난다 → 서버 터미널이 켜져 있는지 봅니다
- ⛔ 설명을 바꿨는데 예전 문구가 나온다 → 2장 셀을 2.1 셀부터 차례로 다시 실행하고, 서버를 Ctrl+C로 끈 뒤 다시 띄웁니다

### 3.3 Researcher에게 위임

**✍️ CHANGE_HERE: Researcher에게 맡길 주제를 바꿔 봅니다**

먼저 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 주제를 바꿔(예: "반도체 공정의 AI 활용 사례") 이 셀을 다시 실행합니다. 서버는 다시 띄우지 않아도 됩니다.

In [ ]:
# ── Researcher에게 조사 위임 ───────────────────────────────────────────────
# ✍️ CHANGE_HERE: Researcher에게 맡길 주제를 바꿔 봅니다
topic = "LangGraph Multi-Agent 시스템의 실제 활용 사례"
print(f'=== Researcher에게 위임: "{topic}" ===')
research_result = await call_researcher(topic)
print(f"리서치 결과:\n{research_result}\n")

**성공 기준**

- 🟢 `=== Researcher에게 위임: "…" ===` 아래에 `리서치 결과:`와 조사한 글이 출력되었다
- 🟢 서버 터미널에 `[search_web] 검색어: …` 줄이 새로 나왔다(모델이 검색 없이 답하면 없을 수 있으니 한 번 더 실행해 봅니다)
- 🟢 주제를 바꾸면 바꾼 주제에 맞는 글이 나온다
- ⛔ 오류가 나거나, `리서치 결과:` 아래가 비어 있거나, 보낸 주제나 검색 결과 한 줄만 그대로 나온다 → 이 셀에서 오류가 났는데 서버 터미널에 새 오류(Traceback)가 없으면 3.1에서 고른 보기를, 그 밖에는 2장에서 고른 보기를 성공 기준과 비교해 다른 보기로 바꿉니다. 2장을 바꿨으면 2.1 셀부터 차례로 다시 실행하고, 서버를 Ctrl+C로 끈 뒤 다시 띄우고 이 셀을 다시 실행합니다. 3.1을 바꿨으면 3.1 셀을 다시 실행한 뒤 이 셀을 다시 실행합니다(서버는 그대로 둡니다)

### 3.4 Writer가 글 작성

In [ ]:
# ── Writer가 리서치 결과로 글 작성 ────────────────────────────────────────
# Writer는 직접 LLM을 쓰지만, 리서치 작업은 Researcher에게 위임했다.
# → 이종 Agent 간 역할 분리가 A2A의 핵심
print("=== Writer가 블로그 포스트 작성 ===")
llm = get_llm()
response = llm.invoke(
    [
        SystemMessage(
            content="너는 전문 작가야. 리서치 결과를 바탕으로 블로그 포스트를 작성해줘."
        ),
        HumanMessage(
            content=f"리서치 결과:\n{research_result}\n\n이 내용으로 블로그 포스트를 작성해줘."
        ),
    ]
)
print(response.content)

**성공 기준**

- 🟢 `=== Writer가 블로그 포스트 작성 ===` 아래에 리서치 결과를 바탕으로 한 블로그 글이 출력되었다
- ⛔ `NameError: name 'research_result' is not defined`가 난다 → 바로 위 셀을 먼저 실행합니다

**볼 지점**
- "Agent Card 확인" 셀은 모델을 호출하지 않습니다. 서버가 공개한 카드(`/.well-known/agent-card.json`)를 읽어 이름·설명·스킬을 보여 줄 뿐입니다.
- "Researcher에게 위임" 셀의 조사는 이 노트북이 아니라 **서버의 Agent**가 합니다. 그래서 검색 기록(`[search_web]` 줄)도 이 노트북이 아니라 서버 터미널에 나옵니다.
- "Writer가 글 작성" 셀은 서버와 관계없이 이 노트북에서 모델을 직접 호출합니다. 결과는 실행할 때마다 조금씩 다릅니다.

### 3.5 AI에게 물어보기

**프롬프트 힌트** "Researcher에게 위임"과 "Writer가 글 작성" 두 셀을 붙여 넣고, 어느 줄이 원격 Agent에게 일을 맡기고 어느 줄이 이 노트북에서 직접 모델을 호출하는지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `await call_researcher(topic)`은 A2A로 서버의 Researcher에게 맡기는 줄이라고 짚었다
- 🟢 `llm.invoke(...)`는 이 노트북에서 모델을 직접 호출하는 줄이라고 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 4. 정리

A2A로 Agent 사이에 일을 나눴습니다.

| 쪽 | 한 일 | 쓴 것 |
|---|---|---|
| 서버 (Researcher) | 요청을 받아 내부 LangGraph Agent를 실행하고 결과를 돌려줌 | AgentExecutor, Agent Card, RequestHandler |
| 클라이언트 (이 노트북) | 카드를 읽고, 조사를 맡기고, 받은 결과로 글을 씀 | A2ACardResolver, SendMessageRequest |

맡기는 쪽은 Researcher의 내부 구현을 몰라도 카드와 메시지 형식만 맞추면 일을 맡길 수 있습니다.

**실습을 마치면 서버 터미널에서 Ctrl+C로 서버를 끕니다.** 같은 8001 포트를 쓰는 다른 서버를 띄울 때 부딪히지 않게 하기 위해서입니다.